# Camada Gold 

## Objetivo

Este notebook é a tarefa `Silver_to_Gold` do Job `v_credit_medallion_pipeline` e depende da execução prévia de `Bronze_to_Silver`.

## Modelo final

```
                               ┌──────────────────┐
                               │   dim_cliente    │
                               └────────┬─────────┘
                                        │ id_cliente
 ┌──────────────────┐       ┌───────────┴───────────────┐       ┌──────────────────┐
 │  dim_atendente   ├───────┤      fato_chamados        ├───────┤    dim_motivo    │
 └──────────────────┘       │  (fato central)           │       └──────────────────┘
          id_atendente      │  1 linha por chamado      │       id_motivo
                            └─────────────┬─────────────┘
                                          │ id_canal
                                 ┌────────┴─────────┐
                                 │    dim_canal     │
                                 └──────────────────┘

 Views setoriais (recortes da fato):  vw_financeiro · vw_operacoes · vw_experiencia_cliente · vw_gestao
 Controle:                            tb_auditoria_gold (1 linha por execução)
```

## Objetos da Gold

| Objeto | Tipo | Grão | Linhas esperadas | Uso principal |
|---|---|---|---|---|
| `fato_chamados` | Fato central | 1 chamado | 131.838 | Análises por chamado, canal, motivo, atendente e período |
| `dim_cliente` | Dimensão | 1 cliente | 47.705 | Perfil (região, idade), **sem dados pessoais** |
| `dim_atendente` | Dimensão | 1 atendente | 25 | Nível e vínculo |
| `dim_motivo` | Dimensão | 1 motivo | 14 | Categoria e criticidade |
| `dim_canal` | Dimensão | 1 canal | 4 | Tipo de canal (autosserviço ou humano) |
| `vw_financeiro`, `vw_operacoes`, `vw_experiencia_cliente`, `vw_gestao` | Views | 1 chamado | 131.838 cada | Recortes setoriais da fato (mesmas colunas da versão anterior) |
| `tb_auditoria_gold` | Controle | 1 execução | 1 | Flags de qualidade e conciliações da execução |

A `fato_chamados` mantém, além das chaves, os atributos descritivos (ex.: `nome_motivo`, `regiao`), o que facilita o uso direto em BI, CSV e nas views. As dimensões permitem filtros e joins no padrão estrela.

## Dicionário de dados

### `fato_chamados` (fato central — 1 linha por chamado)

| Grupo | Colunas | Origem | Observação |
|---|---|---|---|
| Chaves | `id_chamado`, `id_cliente`, `id_atendente`, `id_canal`, `id_motivo` | Silver `tb_chamados`; `id_motivo` via `tb_motivos` | Chaves estrangeiras para as dimensões |
| Horários e datas | `hora_abertura_chamado`, `hora_inicio_atendimento`, `hora_finalizacao_atendimento`, `data_abertura`, `ano_mes_abertura`, `hora_abertura` | Silver `tb_chamados_hora` | Já com ano e virada de dia corrigidos na Silver |
| Canal | `nome_canal`, `tipo_canal`, `status_canal`, `autosservico`, `etapa_canal`, `qtd_etapas` | Silver `tb_canais` | `etapa_canal` = nome do canal em maiúsculas; `qtd_etapas` é sempre nula (a base não registra etapas) e existe apenas por compatibilidade com a `vw_operacoes` |
| Motivo | `nome_motivo`, `categoria`, `criticidade` | Silver `tb_motivos` | |
| Cliente | `regiao`, `idade`, `faixa_idade`, `data_cadastro`, `dia_vencimento_fatura` | Silver `tb_clientes` | Nome e e-mail não entram na Gold |
| Atendente | `nome_atendente`, `nivel_atendimento`, `data_admissao`, `data_desligamento`, `status_atendente` | Silver `tb_atendentes` | Nulos em URA/Chatbot |
| Resultado | `resolvido`, `abandonado` | Silver (`resolvido`) / Gold (`abandonado`) | `resolvido` booleano; `abandonado` nulo em URA/Chatbot |
| Tempos (segundos) | `tempo_espera_segundos`, `tempo_ate_abandono_segundos`, `tempo_atendimento_segundos`, `tempo_total_segundos`, `faixa_espera` | Gold (seção 6), a partir dos horários | |
| Custo | `custo_total`, `qtd_registros_custo`, `qtd_custos_nulos`, `qtd_custos_negativos` | Silver `tb_custos`, agregada por chamado | Chamado sem custo fica com `custo_total` nulo |
| Satisfação | `nota_media`, `qtd_pesquisas`, `qtd_notas_preenchidas`, `qtd_notas_validas`, `qtd_notas_invalidas` | Silver `tb_pesquisa_satisfacao`, agregada por chamado | |
| Recontato | `recontato`, `janela_recontato_dias`, `abertura_contato_anterior`, `segundos_desde_contato_anterior`, `qtd_contatos_anteriores_7d`, `transicao_canal_recontato` | Gold (seção 7) | Indicador aproximado (ver regras de negócio) |
| Qualidade | 20 colunas `flag_*` e `qtd_flags_qualidade` | Gold (seções 5 e 8) | |

### Dimensões

| Dimensão | Colunas | Origem |
|---|---|---|
| `dim_cliente` | `id_cliente`, `regiao`, `idade`, `faixa_idade`, `data_cadastro`, `dia_vencimento_fatura` | Silver `tb_clientes` (sem nome e e-mail) |
| `dim_atendente` | `id_atendente`, `nome_atendente`, `nivel_atendimento`, `data_admissao`, `data_desligamento`, `status_atendente` | Silver `tb_atendentes` |
| `dim_motivo` | `id_motivo`, `nome_motivo`, `categoria`, `criticidade` | Silver `tb_motivos` |
| `dim_canal` | `id_canal`, `nome_canal`, `tipo_canal`, `status_canal`, `autosservico` | Silver `tb_canais` |

### Views setoriais

Todas são recortes da `fato_chamados` (1 linha por chamado), com as mesmas colunas da versão anterior da Gold.

**Colunas comuns às 4 views:** `id_chamado`, `data_abertura`, `ano_mes_abertura`, `id_cliente`, `id_motivo`, `nome_motivo`, `categoria`, `criticidade`, `id_canal`, `nome_canal`, `tipo_canal`, `regiao`, `faixa_idade`, `resolvido`, `abandonado`, `recontato`, `qtd_contatos_anteriores_7d`, `transicao_canal_recontato`, `faixa_espera`, `qtd_flags_qualidade`.

| View | Colunas específicas |
|---|---|
| `vw_financeiro` | `custo_total`, `qtd_registros_custo`, `qtd_custos_nulos`, `qtd_custos_negativos`, `tempo_atendimento_segundos`, `nota_media`, `qtd_notas_validas`, `autosservico`, `flag_custo_ausente`, `flag_custo_invalido` |
| `vw_operacoes` | `id_atendente`, `nome_atendente`, `nivel_atendimento`, `hora_abertura_chamado`, `hora_inicio_atendimento`, `hora_finalizacao_atendimento`, `tempo_espera_segundos`, `tempo_ate_abandono_segundos`, `tempo_atendimento_segundos`, `tempo_total_segundos`, `etapa_canal`, `qtd_etapas`, `flag_etapas_nao_observadas`, `flag_cronologia_invalida`, `flag_atendente_ausente` |
| `vw_experiencia_cliente` | `nota_media`, `qtd_pesquisas`, `qtd_notas_validas`, `qtd_notas_invalidas`, `tempo_espera_segundos`, `tempo_atendimento_segundos`, `abertura_contato_anterior`, `janela_recontato_dias`, `flag_pesquisa_ausente`, `flag_nota_invalida`, `flag_recontato_nao_avaliavel` |
| `vw_gestao` | `custo_total`, `nota_media`, `qtd_notas_validas`, `tempo_espera_segundos`, `tempo_atendimento_segundos`, `autosservico`, `flag_custo_ausente` |

## Regras de negócio

- **Abandono:** chamado de canal humano sem hora de início de atendimento. Não se aplica a URA/Chatbot (valor nulo), pois não há fila.
- **Tempos:** calculados a partir dos horários. A espera de abandonos fica em `tempo_ate_abandono_segundos` (até a desistência), separada da espera de chamados atendidos.
- **Faixa de espera:** considera a espera vivida pelo cliente (até o início do atendimento ou, em abandonos, até a desistência). Faixas descritivas, não metas de SLA.
- **Recontato:** chamado com outro chamado anterior do **mesmo cliente e motivo** aberto nos **7 dias** anteriores. É uma regra analítica, não um campo registrado na base. **Limitação:** cerca de 99% dos casos ocorrem em menos de 1 hora do contato anterior, o que indica sequências de atendimento na mesma interação (ex.: URA seguida de atendimento humano), e não necessariamente o cliente voltando outro dia. O indicador deve ser interpretado como aproximação.
- **Flags de qualidade:** sinalizam ausência, inconsistência ou correção de dados; não excluem chamados das métricas. `qtd_flags_qualidade` soma apenas flags de problema/correção (não conta ausências esperadas, como chamado sem pesquisa).

## Validações executadas

1. Contrato da Silver: tabelas, colunas, tipos e chaves primárias.
2. Grão da fato: uma linha por chamado, sem perder nem duplicar IDs.
3. Conciliação de custo (R$ 1.453.564,18) e de pesquisas com a Silver.
4. Integridade referencial entre a fato e as dimensões.

## Diferenças em relação à versão anterior da Gold

- A tabela central `tb_chamados_central` passa a se chamar `fato_chamados` e ganha as dimensões `dim_cliente`, `dim_atendente`, `dim_motivo` e `dim_canal`.
- As 4 views setoriais continuam com os **mesmos nomes e colunas**, agora lidas da `fato_chamados`.
- Removidas da tabela central as cópias de métricas da Silver que divergiam das métricas da Gold (`abandonado_silver`, `tempo_espera_silver_segundos`, `tempo_atendimento_silver_segundos`, `resolvido_silver`, `motivo_informado_silver`) e a coluna de conferência `id_cliente_horario`. Nenhuma delas era usada pelas views.
- Removidos nome e e-mail dos clientes (dados pessoais não usados na análise).
- `faixa_espera` passa a ser preenchida também para abandonos.
- `qtd_flags_qualidade` deixa de somar a flag constante `flag_etapas_nao_observadas` e as ausências esperadas (ex.: chamado sem pesquisa).

## Parâmetros

| Parâmetro | Padrão | Descrição |
|---|---|---|
| `exportar_csv` | `true` | Exporta a fato, as views e as dimensões em CSV para o Volume `gold/exports` (mesmo comportamento da versão anterior). Use `false` para pular a exportação |

In [0]:
import json
from datetime import datetime, timezone
from functools import reduce
from uuid import uuid4

from pyspark.sql import functions as F
from pyspark.sql import Window

In [0]:
# Configurações
catalog = "v_credit_medallion"
silver_schema_name = "silver"
gold_schema_name = "gold"

# Schemas completos
silver_schema = f"{catalog}.{silver_schema_name}"
gold_schema = f"{catalog}.{gold_schema_name}"

# Parâmetro de negócio
JANELA_RECONTATO_DIAS = 7   # recontato: chamado anterior do mesmo cliente e motivo em até 7 dias

# Exportação de CSVs (ligada por padrão, como na versão anterior; use exportar_csv=false para pular)
dbutils.widgets.text("exportar_csv", "true")
EXPORTAR_CSV = dbutils.widgets.get("exportar_csv").strip().lower() == "true"

# Fuso horário da sessão (registrado no manifesto da exportação)
fuso_horario = spark.conf.get("spark.sql.session.timeZone")

## 1. Leitura e contrato da Silver

Antes de qualquer escrita, a Gold confere se a Silver cumpre o **contrato** abaixo: todas as tabelas existem, têm as colunas e os tipos esperados, e as chaves primárias são preenchidas e únicas. Se algo falhar, a execução é interrompida sem alterar a Gold.

As tabelas lidas são projetadas apenas nas colunas do contrato (metadados técnicos, como `_processed_at`, ficam de fora). Nome e e-mail dos clientes são validados aqui, mas removidos na seção 4.

In [0]:
# Contrato da Silver: colunas e tipos consumidos pela Gold
CONTRATO = {
    "tb_chamados": {
        "id_chamado": "int", "id_cliente": "bigint", "id_atendente": "int", "id_canal": "int",
        "resolvido": "string", "motivo": "string", "tempo_espera": "int",
        "tempo_atendimento": "int", "flg_abandonado": "boolean",
    },
    "tb_chamados_hora": {
        "id_chamado": "int", "id_cliente": "bigint", "hora_abertura_chamado": "timestamp",
        "hora_inicio_atendimento": "timestamp", "hora_finalizacao_atendimento": "timestamp",
        "flg_ano_corrigido": "boolean", "flg_virada_dia": "boolean",
    },
    "tb_custos": {"id_custo": "int", "id_chamado": "int", "custo": "decimal(10,2)"},
    "tb_pesquisa_satisfacao": {"id_pesquisa": "int", "id_chamado": "int", "nota_atendimento": "int"},
    "tb_clientes": {
        "id_cliente": "bigint", "email": "string", "regiao": "string", "idade": "int",
        "data_cadastro": "date", "dia_vencimento_fatura": "int", "nome_cliente": "string",
    },
    "tb_atendentes": {
        "id_atendente": "int", "nome_atendente": "string", "nivel_atendimento": "int",
        "data_admissao": "date", "data_desligamento": "date", "status_atendente": "string",
    },
    "tb_motivos": {"id_motivo": "int", "nome_motivo": "string", "categoria": "string", "criticidade": "string"},
    "tb_canais": {"id_canal": "int", "nome_canal": "string", "tipo_canal": "string", "status_canal": "string"},
}

# Chave primária de cada tabela da Silver
CHAVES = {
    "tb_chamados": "id_chamado", "tb_chamados_hora": "id_chamado", "tb_custos": "id_custo",
    "tb_pesquisa_satisfacao": "id_pesquisa", "tb_clientes": "id_cliente",
    "tb_atendentes": "id_atendente", "tb_motivos": "id_motivo", "tb_canais": "id_canal",
}


def validar_silver(tabelas):
    """
    Interrompe a execução se a Silver não cumprir o contrato
    (coluna faltante, tipo diferente, chave nula ou duplicada).
    Retorna as tabelas projetadas apenas nas colunas do contrato.
    """
    for nome, contrato in CONTRATO.items():
        df = tabelas[nome]

        # Colunas obrigatórias
        faltantes = set(contrato) - set(df.columns)
        if faltantes:
            raise ValueError(f"{nome}: faltam colunas {sorted(faltantes)}")

        # Tipos esperados
        tipos = {campo.name: campo.dataType.simpleString() for campo in df.schema.fields}
        incorretos = {c: (tipos[c], tipo) for c, tipo in contrato.items() if tipos[c] != tipo}
        if incorretos:
            raise ValueError(f"{nome}: tipos (recebido, esperado): {incorretos}")

        # Mantém apenas as colunas de negócio
        df = df.select(*contrato)

        # Chave primária preenchida e única
        chave = CHAVES[nome]
        if df.filter(F.col(chave).isNull()).limit(1).count():
            raise ValueError(f"{nome}: chave {chave} nula")
        if df.groupBy(chave).count().filter("count > 1").limit(1).count():
            raise ValueError(f"{nome}: chave {chave} duplicada; corrija na Silver")

        tabelas[nome] = df
    return tabelas


# Verifica a existência das tabelas antes da leitura
faltantes = [nome for nome in CONTRATO if not spark.catalog.tableExists(f"{silver_schema}.{nome}")]
if faltantes:
    raise ValueError(f"Tabelas Silver ausentes: {faltantes}. Execute Bronze_to_Silver antes desta tarefa.")

# Leitura e validação: t = tabelas da Silver projetadas no contrato
t = validar_silver({nome: spark.table(f"{silver_schema}.{nome}") for nome in CONTRATO})

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {gold_schema}")
print("Contrato da Silver validado.")

## 2. Funções auxiliares

- **`chave_motivo`:** normaliza o nome do motivo (minúsculas, sem acentos e sem espaços extras) para o join entre `tb_chamados` e `tb_motivos`.
- **`faixa_idade`:** faixa etária descritiva, usada na fato e na `dim_cliente` (mesma regra nos dois lugares).
- **`escrever_tabela`:** grava uma tabela Delta na Gold com sobrescrita completa, no mesmo padrão da Silver.

In [0]:
def chave_motivo(coluna):
    """Chave normalizada do motivo: minúsculas, sem acentos e sem espaços extras."""
    return F.regexp_replace(
        F.translate(F.lower(F.trim(coluna)), "áàâãäéèêëíìîïóòôõöúùûüç", "aaaaaeeeeiiiiooooouuuuc"),
        r"\s+",
        " ",
    )


def faixa_idade(coluna):
    """Faixa etária descritiva (idades fora de 18–100 chegam nulas da Silver)."""
    idade = F.col(coluna)
    return (
        F.when(idade.isNull() | ~idade.between(18, 100), "Não informada/inválida")
         .when(idade < 30, "18-29")
         .when(idade < 45, "30-44")
         .when(idade < 60, "45-59")
         .otherwise("60+")
    )


def escrever_tabela(df, nome):
    """Grava uma tabela Delta na Gold com sobrescrita completa."""
    (
        df.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(f"{gold_schema}.{nome}")
    )

## 3. Agregações por chamado (custos e pesquisas)

Custos e pesquisas são **agregados por `id_chamado` antes do join**. A Silver já garante no máximo um custo e uma pesquisa por chamado; a agregação protege o grão da fato (1 linha por chamado) caso essa garantia mude no futuro.

- Chamado sem custo fica com `custo_total` **nulo** (ausência de custo não vira custo zero); apenas as contagens recebem zero.
- `nota_media` considera apenas notas válidas (1 a 5).

In [0]:
# Custos: 1 linha por chamado
df_custo = (
    t["tb_custos"]
    .groupBy("id_chamado")
    .agg(
        F.count("*").alias("qtd_registros_custo"),
        F.sum("custo").alias("custo_total"),
        F.sum(F.col("custo").isNull().cast("int")).alias("qtd_custos_nulos"),
        F.sum((F.col("custo") < 0).cast("int")).alias("qtd_custos_negativos"),
    )
)

# Pesquisas: 1 linha por chamado
df_nota = (
    t["tb_pesquisa_satisfacao"]
    .groupBy("id_chamado")
    .agg(
        F.count("*").alias("qtd_pesquisas"),
        F.count("nota_atendimento").alias("qtd_notas_preenchidas"),
        F.sum(F.col("nota_atendimento").between(1, 5).cast("int")).alias("qtd_notas_validas"),
        F.sum((F.col("nota_atendimento").isNotNull()
               & ~F.col("nota_atendimento").between(1, 5)).cast("int")).alias("qtd_notas_invalidas"),
        F.avg(F.when(F.col("nota_atendimento").between(1, 5), F.col("nota_atendimento"))).alias("nota_media"),
    )
)

## 4. Montagem da fato: joins com a Silver

A base é a `tb_chamados` da Silver, unida por **LEFT JOIN** às demais tabelas. Nenhum chamado é descartado: ausências de correspondência viram flags na seção 5.

| Tabela | Chave do join | Observação |
|---|---|---|
| `tb_chamados_hora` | `id_chamado` | `id_cliente` renomeado para `id_cliente_horario`, apenas para conferência |
| custos e pesquisas (seção 3) | `id_chamado` | Já agregados por chamado |
| `tb_clientes` | `id_cliente` | **Nome e e-mail removidos** (dados pessoais desnecessários para a análise) |
| `tb_atendentes` | `id_atendente` | |
| `tb_motivos` | chave normalizada do nome | Traz `id_motivo`, categoria e criticidade |
| `tb_canais` | `id_canal` | |

Tempos e abandono da Silver são substituídos pelas versões calculadas na seção 6, para que exista **uma única versão de cada métrica** na Gold.


In [0]:
# Horários: id_cliente renomeado para conferir a consistência com tb_chamados
df_hora = (
    t["tb_chamados_hora"]
    .withColumnRenamed("id_cliente", "id_cliente_horario")
    .withColumn("_tem_horario", F.lit(True))
)

# Clientes sem dados pessoais
df_cliente = (
    t["tb_clientes"]
    .drop("nome_cliente", "email")
    .withColumn("_tem_cliente", F.lit(True))
)

df_atendente = t["tb_atendentes"].withColumn("_tem_atendente", F.lit(True))

# Motivos com a chave normalizada usada no join
df_motivo = (
    t["tb_motivos"]
    .withColumn("_chave_motivo", chave_motivo(F.col("nome_motivo")))
    .withColumn("_tem_motivo", F.lit(True))
)

# A chave do motivo precisa ser preenchida e única para não multiplicar chamados no join
if df_motivo.filter(F.col("_chave_motivo").isNull() | (F.col("_chave_motivo") == "")).limit(1).count():
    raise ValueError("tb_motivos: nome vazio/nulo impede o mapeamento de id_motivo")
if df_motivo.groupBy("_chave_motivo").count().filter("count > 1").limit(1).count():
    raise ValueError("tb_motivos: nomes normalizados ambíguos; Gold interrompida para não multiplicar chamados")

df_canal = t["tb_canais"].withColumn("_tem_canal", F.lit(True))

# Base da fato: chamados da Silver
df_fato = (
    t["tb_chamados"]
    # Resolução em booleano (Silver: "Sim" / "Não" / NULL)
    .withColumn(
        "resolvido",
        F.when(F.col("resolvido") == "Sim", True)
         .when(F.col("resolvido") == "Não", False)
    )
    # Chave normalizada para o join com motivos
    .withColumn("_chave_motivo", chave_motivo(F.col("motivo")))
    # Substituídos por nome_motivo e pelas métricas calculadas na seção 6
    .drop("motivo", "tempo_espera", "tempo_atendimento", "flg_abandonado")
)

# LEFT JOINs: nenhum chamado é descartado
for df_dim, chave in [
    (df_hora, "id_chamado"),
    (df_custo, "id_chamado"),
    (df_nota, "id_chamado"),
    (df_cliente, "id_cliente"),
    (df_atendente, "id_atendente"),
    (df_motivo, "_chave_motivo"),
    (df_canal, "id_canal"),
]:
    df_fato = df_fato.join(df_dim, on=chave, how="left")

# Contagens sem correspondência viram 0 (custo_total e nota_media continuam nulos)
CONTAGENS = [
    "qtd_registros_custo", "qtd_custos_nulos", "qtd_custos_negativos",
    "qtd_pesquisas", "qtd_notas_preenchidas", "qtd_notas_validas", "qtd_notas_invalidas",
]
df_fato = df_fato.fillna(0, subset=CONTAGENS)

## 5. Flags de qualidade

Cada flag indica uma ausência, inconsistência ou correção de dado. As flags **não excluem** chamados das métricas; servem para auditoria e para a leitura de confiabilidade.

| Flag | Significado |
|---|---|
| `flag_horario_ausente`, `flag_cliente_ausente`, `flag_motivo_ausente`, `flag_canal_ausente` | Sem correspondência no join |
| `flag_atendente_ausente` | Canal humano sem atendente válido |
| `flag_horario_incompleto`, `flag_cronologia_invalida` | Horários nulos ou fora de ordem |
| `flag_cliente_horario_divergente` | `id_cliente` diferente entre `tb_chamados` e `tb_chamados_hora` |
| `flag_resolucao_ausente` | Resolução desconhecida na origem |
| `flag_custo_ausente`, `flag_custo_invalido` | Chamado sem custo ou com custo nulo/negativo |
| `flag_pesquisa_ausente`, `flag_nota_ausente`, `flag_nota_invalida` | Sem pesquisa, pesquisa sem nota ou nota fora de 1–5 |
| `flag_idade_invalida`, `flag_idade_ausente` | Idade fora de 18–100 ou não informada |
| `flag_ano_corrigido`, `flag_virada_dia` | Correções de horário feitas na Silver (seção 8) |
| `flag_etapas_nao_observadas` | Sempre verdadeira: a base não registra as etapas do atendimento (seção 6) |
| `flag_recontato_nao_avaliavel` | Recontato não calculável (seção 7) |

Várias flags ficam zeradas com a Silver atual; são mantidas como proteção caso a Silver mude.

In [0]:
# Atalhos para os horários
abertura = F.col("hora_abertura_chamado")
inicio = F.col("hora_inicio_atendimento")
fim = F.col("hora_finalizacao_atendimento")

# Horários fora de ordem
cronologia = (inicio < abertura) | (fim < abertura) | (fim < inicio)

df_fato = (
    df_fato
    # Joins sem correspondência
    .withColumn("flag_horario_ausente", F.col("_tem_horario").isNull())
    .withColumn("flag_cliente_ausente", F.col("_tem_cliente").isNull())
    .withColumn("flag_motivo_ausente", F.col("_tem_motivo").isNull())
    .withColumn("flag_canal_ausente", F.col("_tem_canal").isNull())

    # Canal humano sem atendente válido
    .withColumn(
        "flag_atendente_ausente",
        F.coalesce(F.lower(F.col("tipo_canal")) == "humano", F.lit(False)) & F.col("_tem_atendente").isNull()
    )

    # Horários
    .withColumn("flag_horario_incompleto", abertura.isNull() | fim.isNull())
    .withColumn("flag_cronologia_invalida", F.coalesce(cronologia, F.lit(False)))
    .withColumn(
        "flag_cliente_horario_divergente",
        F.coalesce(F.col("id_cliente") != F.col("id_cliente_horario"), F.lit(False))
    )

    # Resultado, custo e pesquisa
    .withColumn("flag_resolucao_ausente", F.col("resolvido").isNull())
    .withColumn("flag_custo_ausente", F.col("qtd_registros_custo") == 0)
    .withColumn("flag_custo_invalido", (F.col("qtd_custos_nulos") > 0) | (F.col("qtd_custos_negativos") > 0))
    .withColumn("flag_pesquisa_ausente", F.col("qtd_pesquisas") == 0)
    .withColumn("flag_nota_ausente", (F.col("qtd_pesquisas") > 0) & (F.col("qtd_notas_preenchidas") == 0))
    .withColumn("flag_nota_invalida", F.col("qtd_notas_invalidas") > 0)

    # Perfil do cliente
    .withColumn("flag_idade_invalida", F.col("idade").isNotNull() & ~F.col("idade").between(18, 100))
    .withColumn("flag_idade_ausente", F.col("idade").isNull())
)

## 6. Métricas de abandono, tempo e atributos descritivos

- **`abandonado`:** chamado humano sem início de atendimento. Nulo em URA/Chatbot, onde não há fila; assim, a média da coluna já é a taxa de abandono sobre os chamados humanos.
- **`tempo_espera_segundos`:** abertura → início, somente chamados atendidos (0 em URA/Chatbot).
- **`tempo_ate_abandono_segundos`:** abertura → desistência, somente abandonos.
- **`tempo_atendimento_segundos`:** início → finalização, somente chamados atendidos.
- **`tempo_total_segundos`:** abertura → finalização.
- **`faixa_espera`:** faixa da espera vivida pelo cliente (até o início ou, em abandonos, até a desistência). Faixas descritivas, não metas de SLA.
- **`qtd_etapas` / `flag_etapas_nao_observadas`:** a base não registra as etapas percorridas pelo cliente; a coluna fica nula e a flag verdadeira (mantidas por compatibilidade com a `vw_operacoes`).

In [0]:
humano = F.lower(F.col("tipo_canal")) == "humano"
autosservico = F.lower(F.col("tipo_canal")).isin("autosservico", "autosserviço")
tempo_valido = ~F.col("flag_horario_incompleto") & ~F.col("flag_cronologia_invalida")

df_fato = (
    df_fato
    # Abandono: chamado humano sem início de atendimento
    .withColumn("abandonado", F.when(tempo_valido & humano, inicio.isNull()))

    # Tempos em segundos a partir dos horários
    .withColumn(
        "tempo_espera_segundos",
        F.when(tempo_valido & inicio.isNotNull(), inicio.cast("long") - abertura.cast("long"))
    )
    .withColumn(
        "tempo_ate_abandono_segundos",
        F.when(F.col("abandonado"), fim.cast("long") - abertura.cast("long"))
    )
    .withColumn(
        "tempo_atendimento_segundos",
        F.when(tempo_valido & inicio.isNotNull(), fim.cast("long") - inicio.cast("long"))
    )
    .withColumn(
        "tempo_total_segundos",
        F.when(tempo_valido, fim.cast("long") - abertura.cast("long"))
    )

    # Faixa da espera vivida: até o início ou, em abandonos, até a desistência
    .withColumn(
        "_espera_vivida",
        F.coalesce(F.col("tempo_espera_segundos"), F.col("tempo_ate_abandono_segundos"))
    )
    .withColumn(
        "faixa_espera",
        F.when(F.col("_espera_vivida") == 0, "Sem espera")
         .when(F.col("_espera_vivida") <= 60, "Até 1 min")
         .when(F.col("_espera_vivida") <= 300, "Acima de 1 até 5 min")
         .when(F.col("_espera_vivida") <= 600, "Acima de 5 até 10 min")
         .when(F.col("_espera_vivida") > 600, "Acima de 10 min")
    )
    .drop("_espera_vivida")

    # Atributos descritivos
    .withColumn("autosservico", autosservico)
    .withColumn("etapa_canal", F.upper(F.col("nome_canal")))
    .withColumn("faixa_idade", faixa_idade("idade"))
    .withColumn("data_abertura", F.to_date(abertura))
    .withColumn("ano_mes_abertura", F.date_format(abertura, "yyyy-MM"))
    .withColumn("hora_abertura", F.hour(abertura))

    # Etapas do atendimento não são registradas na base
    .withColumn("qtd_etapas", F.lit(None).cast("int"))
    .withColumn("flag_etapas_nao_observadas", F.lit(True))
)

## 7. Recontato

**Regra (mesma da versão anterior):** um chamado é **recontato** quando existe um chamado anterior do **mesmo cliente e motivo** aberto nos **7 dias** anteriores (`JANELA_RECONTATO_DIAS`). O histórico considera apenas chamados estritamente anteriores; dois chamados no mesmo instante não definem ordem.

| Coluna | Descrição |
|---|---|
| `recontato` | Existe chamado anterior do mesmo cliente e motivo na janela |
| `abertura_contato_anterior`, `segundos_desde_contato_anterior` | Chamado anterior mais recente e distância até ele |
| `qtd_contatos_anteriores_7d` | Quantidade de chamados anteriores do mesmo cliente e motivo na janela |
| `transicao_canal_recontato` | Canal do contato anterior → canal atual (nula em caso de empate) |
| `flag_recontato_nao_avaliavel` | Recontato não calculável (cliente, motivo ou horário inválido) |

⚠️ **Limitação:** é uma regra analítica, não um campo da base. Valor de referência: 56.972 recontatos (43% dos chamados), dos quais cerca de 99% ocorrem em menos de 1 hora do contato anterior, o que indica sequências de atendimento na mesma interação, e não necessariamente o cliente voltando outro dia.

In [0]:
# Elegível: chamado com cliente, motivo e horários válidos
elegivel = (
    F.col("id_cliente").isNotNull()
    & F.col("id_motivo").isNotNull()
    & abertura.isNotNull()
    & ~F.col("flag_cronologia_invalida")
    & ~F.col("flag_cliente_horario_divergente")
)

df_fato = df_fato.withColumn("_instante_abertura_us", F.expr("unix_micros(hora_abertura_chamado)"))

# Chamados elegíveis do mesmo cliente e motivo no mesmo instante (empates)
w_mesmo_instante = Window.partitionBy("id_cliente", "id_motivo", "hora_abertura_chamado")
df_fato = df_fato.withColumn("_contatos_no_instante", F.sum(elegivel.cast("long")).over(w_mesmo_instante))

# Histórico estritamente anterior do mesmo cliente e motivo
w_ordem = Window.partitionBy("id_cliente", "id_motivo").orderBy("_instante_abertura_us")
w_historico = w_ordem.rangeBetween(Window.unboundedPreceding, -1)
w_janela = w_ordem.rangeBetween(-JANELA_RECONTATO_DIAS * 86400 * 1_000_000, -1)

# Contato anterior: o máximo da struct ordena pela abertura (primeiro campo)
contato = F.struct(
    abertura.alias("abertura"),
    F.col("id_chamado").alias("id"),
    F.col("nome_canal").alias("canal"),
    F.col("_tem_canal").alias("canal_encontrado"),
    F.col("_contatos_no_instante").alias("qtd_no_instante"),
)

df_fato = (
    df_fato
    .withColumn("_contato_anterior", F.max(F.when(elegivel, contato)).over(w_historico))
    .withColumn("abertura_contato_anterior", F.when(elegivel, F.col("_contato_anterior.abertura")))
    .withColumn(
        "segundos_desde_contato_anterior",
        (F.col("_instante_abertura_us") - F.expr("unix_micros(abertura_contato_anterior)")) / F.lit(1_000_000.0)
    )

    # Recontato: chamado anterior do mesmo cliente e motivo dentro da janela
    .withColumn(
        "recontato",
        F.when(elegivel, F.coalesce(
            F.col("segundos_desde_contato_anterior").between(0, JANELA_RECONTATO_DIAS * 86400),
            F.lit(False)
        ))
    )
    .withColumn("janela_recontato_dias", F.lit(JANELA_RECONTATO_DIAS))

    # Intensidade: chamados anteriores do mesmo cliente e motivo na janela
    .withColumn(
        "qtd_contatos_anteriores_7d",
        F.when(elegivel, F.coalesce(F.sum(elegivel.cast("long")).over(w_janela), F.lit(0)))
    )

    # Transição de canal do contato anterior para o atual (nula em caso de empate)
    .withColumn(
        "transicao_canal_recontato",
        F.when(~F.col("recontato"), F.lit("Sem recontato na janela"))
         .when(
            F.col("recontato")
            & (F.col("_contato_anterior.qtd_no_instante") == 1)
            & F.col("_tem_canal").isNotNull()
            & F.col("_contato_anterior.canal_encontrado").isNotNull()
            & F.col("nome_canal").isNotNull()
            & F.col("_contato_anterior.canal").isNotNull(),
            F.concat(F.col("_contato_anterior.canal"), F.lit(" → "), F.col("nome_canal"))
         )
    )
    .withColumn("flag_recontato_nao_avaliavel", F.col("recontato").isNull())
)

## 8. Finalização da fato: flags e seleção de colunas

- As flags vindas da Silver (`flg_ano_corrigido`, `flg_virada_dia`) recebem o prefixo `flag_`, padrão da Gold.
- Flags sem valor (joins sem correspondência) viram `False`.
- **`qtd_flags_qualidade`** soma apenas as flags de problema ou correção de dado. Ausências esperadas (chamado sem pesquisa, pesquisa sem nota) e flags de cálculo (etapas, recontato) não entram na soma.
- A seleção final define a ordem das colunas, igual à do dicionário no início do notebook, e descarta colunas auxiliares (prefixo `_`).

In [0]:
# Prefixo padrão para as flags vindas da Silver
df_fato = (
    df_fato
    .withColumnRenamed("flg_ano_corrigido", "flag_ano_corrigido")
    .withColumnRenamed("flg_virada_dia", "flag_virada_dia")
)

# Flags sem valor viram False
FLAGS = sorted(c for c in df_fato.columns if c.startswith("flag_"))
df_fato = df_fato.fillna(False, subset=FLAGS)

# Flags de problema ou correção de dado
FLAGS_QUALIDADE = [
    "flag_horario_ausente", "flag_horario_incompleto", "flag_cronologia_invalida",
    "flag_cliente_horario_divergente", "flag_cliente_ausente", "flag_atendente_ausente",
    "flag_motivo_ausente", "flag_canal_ausente", "flag_resolucao_ausente",
    "flag_custo_ausente", "flag_custo_invalido", "flag_nota_invalida",
    "flag_idade_invalida", "flag_idade_ausente", "flag_ano_corrigido", "flag_virada_dia",
]
df_fato = df_fato.withColumn(
    "qtd_flags_qualidade",
    reduce(lambda a, b: a + b, [F.col(c).cast("int") for c in FLAGS_QUALIDADE])
)

# Colunas finais, na ordem do dicionário de dados
COLUNAS_FATO = [
    # Chaves
    "id_chamado", "id_cliente", "id_atendente", "id_canal", "id_motivo",
    # Horários e datas
    "hora_abertura_chamado", "hora_inicio_atendimento", "hora_finalizacao_atendimento",
    "data_abertura", "ano_mes_abertura", "hora_abertura",
    # Canal
    "nome_canal", "tipo_canal", "status_canal", "autosservico", "etapa_canal", "qtd_etapas",
    # Motivo
    "nome_motivo", "categoria", "criticidade",
    # Cliente
    "regiao", "idade", "faixa_idade", "data_cadastro", "dia_vencimento_fatura",
    # Atendente
    "nome_atendente", "nivel_atendimento", "data_admissao", "data_desligamento", "status_atendente",
    # Resultado
    "resolvido", "abandonado",
    # Tempos
    "tempo_espera_segundos", "tempo_ate_abandono_segundos", "tempo_atendimento_segundos",
    "tempo_total_segundos", "faixa_espera",
    # Custo
    "custo_total", "qtd_registros_custo", "qtd_custos_nulos", "qtd_custos_negativos",
    # Satisfação
    "nota_media", "qtd_pesquisas", "qtd_notas_preenchidas", "qtd_notas_validas", "qtd_notas_invalidas",
    # Recontato
    "recontato", "janela_recontato_dias", "abertura_contato_anterior",
    "segundos_desde_contato_anterior", "qtd_contatos_anteriores_7d", "transicao_canal_recontato",
    # Qualidade
    *FLAGS, "qtd_flags_qualidade",
]
df_fato = df_fato.select(*COLUNAS_FATO)

## 9. Gravação da `fato_chamados` com validações

A fato é gravada primeiro em uma **tabela temporária**. As validações rodam sobre ela (sem recalcular os joins e janelas), e só depois a tabela final é sobrescrita. Se qualquer validação falhar, a `fato_chamados` anterior permanece intacta e a temporária é removida.

Validações:
1. **Grão:** uma linha por chamado da Silver, sem perder nem duplicar IDs.
2. **Custo:** soma igual à da Silver (R$ 1.453.564,18).
3. **Pesquisas:** quantidade igual à da Silver.

In [0]:
staging = f"{gold_schema}._stg_fato_chamados_{uuid4().hex[:12]}"

try:
    # Materializa o resultado uma única vez (sem cache/persist)
    (
        df_fato.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(staging)
    )
    df_stg = spark.table(staging)
    ids_silver = t["tb_chamados"].select("id_chamado")

    # 1. Grão: uma linha por chamado, sem perder nem duplicar IDs
    total_silver = ids_silver.count()
    total_gold = df_stg.count()
    if total_gold != total_silver or df_stg.select("id_chamado").distinct().count() != total_silver:
        raise ValueError("Gold alterou o grão de uma linha por chamado")
    if ids_silver.exceptAll(df_stg.select("id_chamado")).limit(1).count():
        raise ValueError("Gold perdeu IDs da Silver")

    # 2. Custo conciliado com a Silver
    custo_silver = (
        t["tb_custos"].join(ids_silver, "id_chamado", "left_semi")
        .agg(F.sum("custo")).first()[0]
    )
    custo_gold = df_stg.agg(F.sum("custo_total")).first()[0]
    if custo_silver != custo_gold:
        raise ValueError(f"Custo divergente: Silver={custo_silver}, Gold={custo_gold}")

    # 3. Pesquisas preservadas
    pesquisas_silver = t["tb_pesquisa_satisfacao"].join(ids_silver, "id_chamado", "left_semi").count()
    pesquisas_gold = df_stg.agg(F.sum("qtd_pesquisas")).first()[0] or 0
    if pesquisas_gold != pesquisas_silver:
        raise ValueError(f"Pesquisas divergentes: Silver={pesquisas_silver}, Gold={pesquisas_gold}")

    # Validações aprovadas: grava a tabela final
    escrever_tabela(df_stg, "fato_chamados")
finally:
    spark.sql(f"DROP TABLE IF EXISTS {staging}")

fato = spark.table(f"{gold_schema}.fato_chamados")
print(f"fato_chamados: {total_gold:,} chamados | custo conciliado: R$ {custo_gold}")
display(fato.limit(10))

## 10. Dimensões

As dimensões são gravadas a partir da Silver já validada e descrevem as entidades referenciadas pela fato.

| Dimensão | Chave | Observação |
|---|---|---|
| `dim_cliente` | `id_cliente` | Sem nome e e-mail; `faixa_idade` com a mesma regra da fato |
| `dim_atendente` | `id_atendente` | Nível 1 = Atendimento Inicial; Nível 2 = Atendimento Especializado |
| `dim_motivo` | `id_motivo` | 14 motivos em 5 categorias |
| `dim_canal` | `id_canal` | `autosservico` = URA e Chatbot |

Ao final, a **integridade referencial** é verificada: toda chave preenchida na fato precisa existir na dimensão correspondente.

In [0]:
DIMENSOES = {
    # Perfil do cliente, sem dados pessoais
    "dim_cliente": (
        t["tb_clientes"]
        .select(
            "id_cliente", "regiao", "idade", faixa_idade("idade").alias("faixa_idade"),
            "data_cadastro", "dia_vencimento_fatura",
        )
    ),
    "dim_atendente": (
        t["tb_atendentes"]
        .select(
            "id_atendente", "nome_atendente", "nivel_atendimento",
            "data_admissao", "data_desligamento", "status_atendente",
        )
    ),
    "dim_motivo": t["tb_motivos"].select("id_motivo", "nome_motivo", "categoria", "criticidade"),
    "dim_canal": (
        t["tb_canais"]
        .select(
            "id_canal", "nome_canal", "tipo_canal", "status_canal",
            F.lower(F.col("tipo_canal")).isin("autosservico", "autosserviço").alias("autosservico"),
        )
    ),
}

for nome, df_dim in DIMENSOES.items():
    escrever_tabela(df_dim, nome)
    print(f"{nome}: {spark.table(f'{gold_schema}.{nome}').count():,} linhas")

# Integridade referencial: toda chave preenchida na fato existe na dimensão
for coluna, dimensao in [
    ("id_cliente", "dim_cliente"),
    ("id_atendente", "dim_atendente"),
    ("id_motivo", "dim_motivo"),
    ("id_canal", "dim_canal"),
]:
    orfaos = (
        fato.filter(F.col(coluna).isNotNull()).select(coluna).distinct()
        .join(spark.table(f"{gold_schema}.{dimensao}").select(coluna), coluna, "left_anti")
        .count()
    )
    if orfaos:
        raise ValueError(f"fato_chamados.{coluna}: {orfaos} valores sem correspondência em {dimensao}")

print("Integridade referencial entre fato e dimensões validada.")

## 11. Auditoria da execução

A `tb_auditoria_gold` registra, para a execução atual, a quantidade de chamados, o total de cada flag, os registros da Silver sem chamado correspondente (órfãos, contabilizados e nunca incorporados) e as conciliações de custo e pesquisas.

In [0]:
# Registros da Silver sem chamado correspondente
ids_chamados = t["tb_chamados"].select("id_chamado")
auditoria_orfaos = {
    nome: t[nome].join(ids_chamados, "id_chamado", "left_anti").count()
    for nome in ["tb_chamados_hora", "tb_custos", "tb_pesquisa_satisfacao"]
}

flags_fato = sorted(c for c in fato.columns if c.startswith("flag_"))
auditoria = fato.agg(
    F.count("*").alias("qtd_chamados"),
    *[F.sum(F.col(c).cast("long")).alias(c) for c in flags_fato],
)

for nome, qtd in auditoria_orfaos.items():
    auditoria = auditoria.withColumn(f"orfaos_{nome}", F.lit(qtd))

auditoria = (
    auditoria
    .withColumn("custo_total_conciliado", F.lit(custo_gold))
    .withColumn("qtd_pesquisas_silver", F.lit(pesquisas_silver))
    .withColumn("executado_em", F.current_timestamp())
)

escrever_tabela(auditoria, "tb_auditoria_gold")
display(spark.table(f"{gold_schema}.tb_auditoria_gold"))

## 12. Remoção de objetos substituídos

A antiga tabela central `tb_chamados_central` foi substituída pela `fato_chamados` e é removida para não deixar uma cópia desatualizada na Gold. Também são removidos, se existirem, objetos de testes do modelo de jornada (`fato_jornada`, `tb_kpis_mensais`), que não fazem parte deste modelo.

As views setoriais **não** são removidas: elas são recriadas na seção 13 com os mesmos nomes e colunas.

In [0]:
for tabela in ["tb_chamados_central", "fato_jornada", "tb_kpis_mensais"]:
    spark.sql(f"DROP TABLE IF EXISTS {gold_schema}.{tabela}")

print("Objetos substituídos removidos.")

## 13. Views setoriais

As 4 views são recriadas com os **mesmos nomes e colunas da versão anterior**, agora lidas da `fato_chamados`. São projeções simples (sem joins, filtros ou novos cálculos): todas têm 1 linha por chamado.

| View | Público |
|---|---|
| `vw_financeiro` | Custos, resolução e satisfação |
| `vw_operacoes` | Atendentes, horários, tempos e abandono |
| `vw_experiencia_cliente` | Pesquisas, notas, espera e recontato |
| `vw_gestao` | Visão resumida de custo, nota e tempos |

In [0]:
# Colunas comuns às 4 views
COMUNS = [
    "id_chamado", "data_abertura", "ano_mes_abertura", "id_cliente", "id_motivo", "nome_motivo",
    "categoria", "criticidade", "id_canal", "nome_canal", "tipo_canal", "regiao", "faixa_idade", "resolvido",
    "abandonado", "recontato", "qtd_contatos_anteriores_7d", "transicao_canal_recontato",
    "faixa_espera", "qtd_flags_qualidade",
]

# Colunas de cada view (mesmas da versão anterior)
SETORES = {
    "vw_financeiro": COMUNS + [
        "custo_total", "qtd_registros_custo", "qtd_custos_nulos", "qtd_custos_negativos",
        "tempo_atendimento_segundos", "nota_media", "qtd_notas_validas", "autosservico",
        "flag_custo_ausente", "flag_custo_invalido",
    ],
    "vw_operacoes": COMUNS + [
        "id_atendente", "nome_atendente", "nivel_atendimento", "hora_abertura_chamado",
        "hora_inicio_atendimento", "hora_finalizacao_atendimento", "tempo_espera_segundos",
        "tempo_ate_abandono_segundos", "tempo_atendimento_segundos", "tempo_total_segundos", "etapa_canal",
        "qtd_etapas", "flag_etapas_nao_observadas", "flag_cronologia_invalida", "flag_atendente_ausente",
    ],
    "vw_experiencia_cliente": COMUNS + [
        "nota_media", "qtd_pesquisas", "qtd_notas_validas", "qtd_notas_invalidas",
        "tempo_espera_segundos", "tempo_atendimento_segundos", "abertura_contato_anterior",
        "janela_recontato_dias", "flag_pesquisa_ausente", "flag_nota_invalida", "flag_recontato_nao_avaliavel",
    ],
    "vw_gestao": COMUNS + [
        "custo_total", "nota_media", "qtd_notas_validas", "tempo_espera_segundos",
        "tempo_atendimento_segundos", "autosservico", "flag_custo_ausente",
    ],
}

for nome, colunas in SETORES.items():
    projecao = ", ".join(f"`{c}`" for c in colunas)
    spark.sql(f"CREATE OR REPLACE VIEW {gold_schema}.{nome} AS SELECT {projecao} FROM {gold_schema}.fato_chamados")
    print(f"{nome}: {len(colunas)} colunas")

## 14. Exportação CSV

Executada quando `exportar_csv=true` (padrão). Exporta a fato, as 4 views e as dimensões para o Volume `gold/exports`, com os mesmos nomes de arquivo da versão anterior (`gold_completa.csv`, `financeiro.csv`, `operacoes.csv`, `experiencia_cliente.csv`, `gestao.csv`), mais um arquivo por dimensão, em uma pasta por execução (uma falha não sobrescreve uma exportação anterior completa).

- Formato: separador `;`, UTF-8, ponto decimal, todos os campos entre aspas.
- Cada arquivo é relido com o mesmo schema para confirmar que está completo.
- O arquivo `_SUCCESS.json` só é criado quando todos os arquivos da execução foram concluídos.

In [0]:
# Nome do arquivo -> objeto da Gold (mesmos nomes de arquivo da versão anterior + dimensões)
OBJETOS_EXPORTACAO = {
    "gold_completa": "fato_chamados",
    "financeiro": "vw_financeiro",
    "operacoes": "vw_operacoes",
    "experiencia_cliente": "vw_experiencia_cliente",
    "gestao": "vw_gestao",
    "dim_cliente": "dim_cliente",
    "dim_atendente": "dim_atendente",
    "dim_motivo": "dim_motivo",
    "dim_canal": "dim_canal",
}

if EXPORTAR_CSV:
    spark.sql(f"CREATE VOLUME IF NOT EXISTS {gold_schema}.exports")

    # Pasta exclusiva da execução
    execucao = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ") + "_" + uuid4().hex[:8]
    pasta = f"/Volumes/{catalog}/{gold_schema_name}/exports/{execucao}"
    dbutils.fs.mkdirs(pasta)

    manifesto = {
        "execucao_utc": execucao, "separador": ";", "encoding": "UTF-8",
        "decimal": ".", "timezone": fuso_horario, "arquivos": [],
    }

    total_fato = spark.table(f"{gold_schema}.fato_chamados").count()

    for nome_arquivo, objeto in OBJETOS_EXPORTACAO.items():
        df = spark.table(f"{gold_schema}.{objeto}")
        total = df.count()

        # Fato e views têm 1 linha por chamado
        if not objeto.startswith("dim_") and total != total_fato:
            raise ValueError(f"{objeto}: {total} linhas, esperado {total_fato}")

        temporario = f"{pasta}/_tmp_{nome_arquivo}"
        destino = f"{pasta}/{nome_arquivo}.csv"

        # Um único arquivo por tabela
        (
            df.coalesce(1).write.mode("overwrite")
            .option("header", True).option("sep", ";").option("encoding", "UTF-8")
            .option("quoteAll", True).option("quote", '"').option("escape", '"')
            .option("nullValue", "").option("timestampFormat", "yyyy-MM-dd HH:mm:ss.SSSSSS")
            .csv(temporario)
        )
        partes = [
            f.path for f in dbutils.fs.ls(temporario)
            if f.name.startswith("part-") and f.name.endswith(".csv")
        ]
        if len(partes) != 1:
            raise ValueError(f"Esperava um CSV em {temporario}; encontrados {len(partes)}")
        if not dbutils.fs.mv(partes[0], destino):
            raise IOError(f"Não foi possível publicar {destino}")

        # Releitura tipada: detecta quebra de aspas/linhas e confirma o total exportado
        lido = (
            spark.read.schema(df.schema)
            .option("header", True).option("sep", ";").option("quote", '"').option("escape", '"')
            .option("multiLine", True).option("mode", "FAILFAST").option("nullValue", "")
            .option("timestampFormat", "yyyy-MM-dd HH:mm:ss.SSSSSS")
            .csv(destino)
        )
        if lido.count() != total:
            raise ValueError(f"CSV incompleto: {destino}")
        dbutils.fs.rm(temporario, True)

        manifesto["arquivos"].append({
            "arquivo": destino, "objeto": f"{gold_schema}.{objeto}",
            "linhas": total, "colunas": len(df.columns), "schema": df.schema.jsonValue(),
        })
        print(f"EXPORTADO: {destino} | {total:,} linhas | {len(df.columns)} colunas")

    # Marca de conclusão de todos os arquivos da execução
    dbutils.fs.put(f"{pasta}/_SUCCESS.json", json.dumps(manifesto, ensure_ascii=False, indent=2), True)
    print(f"Exportação completa: Catalog → {catalog} → {gold_schema_name} → Volumes → exports → {execucao}")
else:
    print("Exportação CSV desativada (exportar_csv=false). As tabelas Delta continuam disponíveis para consulta.")

## 15. Conferência final

Contagem de linhas de todos os objetos da Gold, um exemplo de consulta no modelo estrela (fato + dimensões) e o exemplo de consulta da versão anterior, sobre a `vw_financeiro`.

In [0]:
for nome in ["fato_chamados", "dim_cliente", "dim_atendente", "dim_motivo", "dim_canal",
             "vw_financeiro", "vw_operacoes", "vw_experiencia_cliente", "vw_gestao", "tb_auditoria_gold"]:
    df = spark.table(f"{gold_schema}.{nome}")
    print(f"{nome}: {df.count():,} linhas | {len(df.columns)} colunas")

In [0]:
# Exemplo de consulta no modelo estrela: custo e volume por categoria de motivo e canal
display(spark.sql(f"""
    SELECT
        m.categoria,
        c.nome_canal,
        COUNT(*)            AS qtd_chamados,
        SUM(f.custo_total)  AS custo_total,
        AVG(f.nota_media)   AS nota_media
    FROM {gold_schema}.fato_chamados f
    JOIN {gold_schema}.dim_motivo m ON f.id_motivo = m.id_motivo
    JOIN {gold_schema}.dim_canal  c ON f.id_canal  = c.id_canal
    GROUP BY m.categoria, c.nome_canal
    ORDER BY custo_total DESC
"""))

In [0]:
# Exemplo da versão anterior: custo e intensidade de contatos por canal na vw_financeiro
display(spark.sql(f"""
    SELECT
        nome_canal,
        COUNT(*)                        AS chamados,
        COUNT(custo_total)              AS chamados_com_custo,
        SUM(custo_total)                AS custo_total,
        AVG(qtd_contatos_anteriores_7d) AS media_contatos_anteriores_7d
    FROM {gold_schema}.vw_financeiro
    GROUP BY nome_canal
"""))